# Real latency-aware LLM routing on Colab

**Question.** Can a prompt-only router reduce warm inference latency while retaining at least 98% of the strongest model's mean accuracy?

This is the real experiment: it downloads reference-scored tasks, runs every prompt through every candidate, measures synchronized GPU generation, caches each answer to Google Drive, and evaluates a router on a sealed test split.

Candidate pool:

- `Qwen/Qwen2.5-7B-Instruct` in 4-bit: strong fallback;
- `Qwen/Qwen2.5-1.5B-Instruct`: matched small autoregressive control;
- `Efficient-Large-Model/Fast_dLLM_v2_1.5B`: block-diffusion candidate.

Start a **GPU Colab runtime** and keep the same seed and decoding settings. The default development run uses 100 prompts per task (300 prompts and 900 model generations), while reusing completed pilot rows. Audit the cached Fast-dLLM responses before enabling the expanded benchmark. All candidates use Fast-dLLM v2's official Transformers 4.53.1 runtime; the notebook no longer patches RoPE, tied weights, or cache internals.

In [1]:
# Gradio is unused here, and Colab's preinstalled release requires Hub 1.x;
# remove it before installing Fast-dLLM's Transformers-4-compatible Hub client.
%pip uninstall -q -y gradio gradio-client
%pip install -q -U "transformers==4.53.1" \
    "huggingface-hub==0.36.2" accelerate "bitsandbytes>=0.46" \
    "datasets>=3.0" pyarrow scikit-learn seaborn "ipywidgets>=8"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 78.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 34.7 MB/s eta 0:00:00


## 1. Configuration and runtime

Results persist in Drive, so a disconnected runtime can resume. Set `MODELS_TO_RUN` to one model key if you prefer one model per Colab session. Latency is **warm batch-size-one generation latency**; model loading is reported separately.

In [2]:

from __future__ import annotations

import gc, hashlib, html as html_lib, json, math, os, random, re, time, warnings
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn.functional as F
import transformers
from IPython.display import HTML, clear_output, display
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader

SEED = 42
N_PER_TASK = 300                # 300 prompts total; raise to 300/task after the audit passes
MAX_INPUT_TOKENS = 1024
MAX_NEW_TOKENS = 192
WARMUP_PROMPTS = 2
FLUSH_EVERY = 5
MIN_QUALITY_RETENTION = 0.98
ORACLE_EPSILON = 0.0            # exact-match quality is binary
USE_GOOGLE_DRIVE = True
FAST_AUDIT_SAMPLE_PER_TASK = 5
FAST_REPAIR_PROMPTS_PER_TASK = 5
RUN_FAST_DLLM_REPAIR_PILOT = True
RUN_EXPANDED_BENCHMARK = True   # set False only when auditing Fast-dLLM before expansion

# End-to-end ModernBERT fine-tuning. The small physical batch is safe on a T4;
# gradient accumulation supplies an effective batch of 16. The lower learning
# rates and cosine decay are deliberately conservative for the small dataset.
ROUTER_BATCH_SIZE = 4
ROUTER_GRADIENT_ACCUMULATION = 4
ROUTER_MAX_EPOCHS = 20
ROUTER_MIN_EPOCHS = 8
ROUTER_EARLY_STOPPING_PATIENCE = 5
ROUTER_EARLY_STOPPING_MIN_DELTA = 1e-4
ROUTER_ENCODER_LR = 1e-5
ROUTER_HEAD_LR = 1e-4
ROUTER_WEIGHT_DECAY = 0.01
ROUTER_WARMUP_RATIO = 0.05
ROUTER_MAX_GRAD_NORM = 1.0
ROUTER_QUALITY_LOSS_WEIGHT = 1.0
ROUTER_TOKEN_LOSS_WEIGHT = 0.20
ROUTER_POLICY_LOSS_WEIGHT = 1.0
SOFT_ORACLE_LATENCY_WEIGHT = 0.10
SOFT_ORACLE_TEMPERATURE = 0.15

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
assert torch.cuda.is_available(), "Choose Runtime > Change runtime type > GPU in Colab."
assert transformers.__version__ == "4.53.1", (
    f"Expected transformers 4.53.1, found {transformers.__version__}. "
    "Restart the Colab runtime after running the install cell, then run top to bottom."
)

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/llm_router_real")
else:
    ROOT = Path("/content/llm_router_real")

for folder in [ROOT / "data", ROOT / "cache", ROOT / "reports"]:
    folder.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = "/content/hf_cache"  # fast local model cache; outcomes stay in Drive
GPU_NAME = torch.cuda.get_device_name(0)
COMPUTE_DTYPE = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
print({"gpu": GPU_NAME, "dtype": str(COMPUTE_DTYPE), "root": str(ROOT)})
print({"transformers": transformers.__version__, "torch": torch.__version__})

Mounted at /content/drive
{'gpu': 'Tesla T4', 'dtype': 'torch.float16', 'root': '/content/drive/MyDrive/llm_router_real'}
{'transformers': '4.53.1', 'torch': '2.11.0+cu128'}


In [3]:
@dataclass(frozen=True)
class ModelSpec:
    repo: str
    kind: str
    four_bit: bool = False

MODELS = {
    "qwen2.5-1.5b-ar": ModelSpec("Qwen/Qwen2.5-1.5B-Instruct", "causal"),
    "fast-dllm-v2-1.5b": ModelSpec(
        "Efficient-Large-Model/Fast_dLLM_v2_1.5B", "fast_dllm"
    ),
    "qwen2.5-7b-4bit": ModelSpec("Qwen/Qwen2.5-7B-Instruct", "causal", four_bit=True),
}
MODELS_TO_RUN = list(MODELS)    # or, for example: ["fast-dllm-v2-1.5b"]

run_manifest = {
    "seed": SEED, "n_per_task": N_PER_TASK,
    "max_input_tokens": MAX_INPUT_TOKENS, "max_new_tokens": MAX_NEW_TOKENS,
    "gpu": GPU_NAME, "dtype": str(COMPUTE_DTYPE),
    "models": {k: vars(v) for k, v in MODELS.items()},
}
manifest_path = ROOT / "run_manifest.json"
if manifest_path.exists():
    old = json.loads(manifest_path.read_text())
    # Model keys map to separate cache files, so changing the candidate pool is
    # safe. This replaces the Transformers-5-only fallback without mixing rows.
    immutable_keys = set(run_manifest) - {"n_per_task", "models"}
    changed = {key: (old.get(key), run_manifest[key]) for key in immutable_keys if old.get(key) != run_manifest[key]}
    assert not changed, f"Configuration changed: {changed}. Use a new ROOT or restore the old settings."
    old_n = int(old.get("n_per_task", 0))
    assert old_n <= N_PER_TASK, "Reducing N_PER_TASK in an existing ROOT is not supported."
    if old_n < N_PER_TASK:
        print(f"Extending the stored experiment from {old_n} to {N_PER_TASK} prompts per task.")
manifest_path.write_text(json.dumps(run_manifest, indent=2))
display(pd.DataFrame.from_dict({k: vars(v) for k, v in MODELS.items()}, orient="index"))

,repo,kind,four_bit
qwen2.5-1.5b-ar,Qwen/Qwen2.5-1.5B-Instruct,causal,False
fast-dllm-v2-1.5b,Efficient-Large-Model/Fast_dLLM_v2_1.5B,fast_dllm,False
qwen2.5-7b-4bit,Qwen/Qwen2.5-7B-Instruct,causal,True


## 2. Reference-scored data

RouterBench's pickle contains prompts plus **previous models'** responses and scores, but not the reference labels needed to score new responses. We therefore load three of its underlying benchmark families directly: GSM8K, MMLU, and ARC-Challenge. This gives the new candidates genuine, reproducible labels rather than borrowing RouterBench's old outcomes.

In [4]:
from datasets import load_dataset

LABELS = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"

def stable_id(task: str, prompt: str) -> str:
    return hashlib.sha1(f"{task}\0{prompt}".encode()).hexdigest()[:16]

def build_prompt_pool() -> pd.DataFrame:
    """Build the complete labeled pool; sampling happens after existing rows are retained."""
    rows = []

    gsm = load_dataset("openai/gsm8k", "main", split="test").to_pandas()
    for row in gsm.itertuples():
        prompt = (
            f"Solve the problem. Show concise reasoning, then end with JSON {{\"answer\": \"number\"}}.\n\n"
            f"{row.question}"
        )
        reference = re.findall(r"-?\d[\d,]*(?:\.\d+)?", row.answer)[-1].replace(",", "")
        rows.append({"task": "gsm8k", "prompt": prompt, "reference": reference})

    mmlu = load_dataset("cais/mmlu", "all", split="test").to_pandas()
    for row in mmlu.itertuples():
        choices = "\n".join(f"{LABELS[i]}. {text}" for i, text in enumerate(row.choices))
        prompt = (
            "Choose the best answer. End with JSON {\"answer\": \"LETTER\"}.\n\n"
            f"Question: {row.question}\n{choices}"
        )
        rows.append({"task": "mmlu", "prompt": prompt, "reference": LABELS[int(row.answer)]})

    arc = load_dataset("allenai/ai2_arc", "ARC-Challenge", split="test").to_pandas()
    for row in arc.itertuples():
        # Hugging Face/Pandas may materialize these nested sequences as NumPy
        # arrays in Colab, so normalize them before using list operations.
        choice_labels = [str(value) for value in row.choices["label"]]
        choice_text = [str(value) for value in row.choices["text"]]
        answer_key = str(row.answerKey)
        if answer_key not in choice_labels:
            raise ValueError(
                f"ARC answer key {answer_key!r} is absent from labels {choice_labels!r}."
            )
        answer_index = choice_labels.index(answer_key)
        choices = "\n".join(f"{LABELS[i]}. {text}" for i, text in enumerate(choice_text))
        prompt = (
            "Choose the best answer. End with JSON {\"answer\": \"LETTER\"}.\n\n"
            f"Question: {row.question}\n{choices}"
        )
        rows.append({"task": "arc_challenge", "prompt": prompt, "reference": LABELS[answer_index]})

    result = pd.DataFrame(rows)
    result.insert(0, "prompt_id", [stable_id(t, p) for t, p in zip(result.task, result.prompt)])
    conflicting = result.groupby("prompt_id").reference.nunique().gt(1)
    assert not conflicting.any(), "The prompt pool contains duplicate prompts with conflicting references."
    result = result.drop_duplicates("prompt_id", keep="first").reset_index(drop=True)
    assert result.prompt_id.is_unique and not result.isna().any().any()
    return result

def extend_prompts(existing: pd.DataFrame, n_per_task: int) -> pd.DataFrame:
    """Retain cached prompt IDs and deterministically append new labeled prompts."""
    required_columns = {"prompt_id", "task", "prompt", "reference"}
    if not existing.empty:
        missing_columns = required_columns - set(existing.columns)
        if missing_columns:
            raise ValueError(f"Stored prompts are missing columns: {sorted(missing_columns)}")
        counts = existing.groupby("task").size()
        if counts.gt(n_per_task).any():
            raise ValueError("Stored prompts exceed N_PER_TASK; use a new ROOT to reduce the dataset.")

    pool = build_prompt_pool()
    parts = []
    task_seeds = {"gsm8k": SEED, "mmlu": SEED + 1, "arc_challenge": SEED + 2}
    for task, task_seed in task_seeds.items():
        kept = existing.loc[existing.task.eq(task)].copy() if not existing.empty else existing.copy()
        needed = n_per_task - len(kept)
        candidates = pool.loc[
            pool.task.eq(task) & ~pool.prompt_id.isin(set(existing.prompt_id))
        ].copy()
        if needed > len(candidates):
            raise ValueError(f"Requested {needed} additional {task} prompts from {len(candidates)} candidates.")
        candidates["_priority"] = [
            hashlib.sha1(f"{task_seed}\0{prompt_id}".encode()).hexdigest()
            for prompt_id in candidates.prompt_id
        ]
        additions = candidates.sort_values("_priority").head(needed).drop(columns="_priority")
        parts.extend([kept, additions])

    result = pd.concat(parts, ignore_index=True)
    assert len(result) == 3 * n_per_task and result.prompt_id.is_unique
    return result

prompts_path = ROOT / "data" / "prompts.parquet"
stored_prompts = pd.read_parquet(prompts_path) if prompts_path.exists() else pd.DataFrame(
    columns=["prompt_id", "task", "prompt", "reference"]
)
stored_counts = stored_prompts.groupby("task").size() if not stored_prompts.empty else pd.Series(dtype=int)
if any(int(stored_counts.get(task, 0)) != N_PER_TASK for task in ["gsm8k", "mmlu", "arc_challenge"]):
    prompts = extend_prompts(stored_prompts, N_PER_TASK)
    prompts.to_parquet(prompts_path, index=False)
else:
    prompts = stored_prompts

display(prompts.groupby("task").size().rename("prompts").to_frame())
display(prompts.head(2))

,prompts
task,
arc_challenge,300
gsm8k,300
mmlu,300


,prompt_id,task,prompt,reference
0,58cf8075de61241b,gsm8k,"Solve the problem. Show concise reasoning, the...",36
1,9de6593ad5d5b36d,gsm8k,"Solve the problem. Show concise reasoning, the...",189


## 3. Real model adapters

Fast-dLLM runs with its repository's custom `generate()` method in the exact
Transformers version pinned by the official implementation. The two Qwen2.5
controls use standard greedy decoding with the same output cap; quantization is
applied only to the 7B fallback. No Transformers internals are monkey-patched.

In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

SYSTEM_PROMPT = "You are a careful benchmark assistant. Follow the requested final-answer format exactly."
FAST_DLLM_REVISION = "25093b6f63300adfd57f72145083c8a528fe4f16"
FAST_DLLM_GENERATION_PROFILE = (
    f"official-tf-4.53.1@{FAST_DLLM_REVISION[:12]}-block32-small8-threshold0.9"
)

class LocalAdapter:
    def __init__(self, name: str, spec: ModelSpec):
        self.name, self.spec = name, spec
        quant = None
        if spec.four_bit:
            quant = BitsAndBytesConfig(
                load_in_4bit=True, bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=COMPUTE_DTYPE, bnb_4bit_use_double_quant=True,
            )

        started = time.perf_counter()
        is_fast_dllm = spec.kind == "fast_dllm"
        revision = FAST_DLLM_REVISION if is_fast_dllm else None
        self.codec = AutoTokenizer.from_pretrained(
            spec.repo, trust_remote_code=is_fast_dllm, revision=revision
        )
        load_kwargs = {
            "device_map": "auto", "torch_dtype": COMPUTE_DTYPE,
            "trust_remote_code": is_fast_dllm, "revision": revision,
            "low_cpu_mem_usage": True,
        }
        if quant is not None:
            load_kwargs["quantization_config"] = quant
        if is_fast_dllm:
            print(f"Loading {spec.repo} in its official Transformers 4.53.1 runtime.")
        self.model = AutoModelForCausalLM.from_pretrained(spec.repo, **load_kwargs)
        self.model.eval()
        self.load_time_s = time.perf_counter() - started

    def encode(self, prompt: str):
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ]
        text = self.codec.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = self.codec(text, return_tensors="pt", truncation=True, max_length=MAX_INPUT_TOKENS)
        return {k: v.to(self.model.device) for k, v in inputs.items()}

    @torch.inference_mode()
    def generate(self, prompt: str) -> dict:
        encode_started = time.perf_counter()
        inputs = self.encode(prompt)
        encode_s = time.perf_counter() - encode_started
        input_tokens = int(inputs["input_ids"].shape[1])

        torch.cuda.reset_peak_memory_stats(); torch.cuda.synchronize()
        started = time.perf_counter()
        if self.spec.kind == "fast_dllm":
            output = self.model.generate(
                inputs["input_ids"], tokenizer=self.codec,
                max_new_tokens=MAX_NEW_TOKENS, small_block_size=8, threshold=0.9,
            )
        else:
            output = self.model.generate(
                **inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                pad_token_id=getattr(self.codec, "pad_token_id", None),
            )
        torch.cuda.synchronize()
        generation_s = time.perf_counter() - started

        new_ids = output[0, input_tokens:]
        response = self.codec.decode(new_ids, skip_special_tokens=True).strip()
        output_tokens = int(new_ids.numel())
        return {
            "response": response, "encode_s": encode_s,
            "generation_s": generation_s, "input_tokens": input_tokens,
            "output_tokens": output_tokens,
            "tokens_per_second": output_tokens / max(generation_s, 1e-9),
            "peak_vram_gb": torch.cuda.max_memory_allocated() / 2**30,
        }

def unload(adapter):
    if hasattr(adapter, "model"):
        del adapter.model
    if hasattr(adapter, "codec"):
        del adapter.codec
    gc.collect(); torch.cuda.empty_cache(); torch.cuda.synchronize()

## 4. Resumable warm benchmarking

First audit the already cached Fast-dLLM responses. This separates a model or
decoding problem from a router problem before the expanded run spends GPU time.
The next cell shows full sampled responses, parser coverage, exact-match quality,
token-cap frequency, and latency, then writes a reusable audit table.

### 4.1 Cache and audit helpers

In [6]:
def cache_path(model_name: str) -> Path:
    return ROOT / "cache" / f"{model_name}.parquet"

def extract_answer(response: str, task: str) -> str | None:
    text = str(response)
    json_answers = re.findall(r'["\']answer["\']\s*:\s*["\']?([^"\'\n}\s]+)', text, flags=re.I)
    if task == "gsm8k":
        candidates = json_answers or re.findall(r"-?\d[\d,]*(?:\.\d+)?", text)
        return candidates[-1].replace(",", "") if candidates else None
    if json_answers:
        match = re.search(r"[A-Z]", json_answers[-1].upper())
        return match.group(0) if match else None
    choices = re.findall(r"(?:^|\W)([A-F])(?:\W|$)", text.upper())
    return choices[-1] if choices else None

def is_degenerate_response(response: str) -> bool:
    """Flag the low-diversity repetition produced by an incompatible runtime."""
    words = re.findall(r"\w+", str(response).lower())
    lines = [line.strip().lower() for line in str(response).splitlines() if line.strip()]
    low_word_diversity = len(words) >= 24 and len(set(words)) / len(words) < 0.20
    repeated_lines = len(lines) >= 8 and len(set(lines)) / len(lines) < 0.40
    return low_word_diversity or repeated_lines

def audit_fast_dllm_cache(sample_per_task: int = FAST_AUDIT_SAMPLE_PER_TASK) -> pd.DataFrame:
    path = cache_path("fast-dllm-v2-1.5b")
    if not path.exists():
        print("No Fast-dLLM cache exists yet. Run a small pilot before expanding the dataset.")
        return pd.DataFrame()

    cached = pd.read_parquet(path).drop_duplicates(["prompt_id", "model"], keep="last")
    required = {"prompt_id", "task", "model", "status", "response"}
    missing_columns = required - set(cached.columns)
    if missing_columns:
        raise ValueError(f"Fast-dLLM cache is missing columns: {sorted(missing_columns)}")

    successful = cached.status.eq("ok")
    if "generation_profile" not in cached:
        cached["generation_profile"] = None
    current_profile = cached.generation_profile.eq(FAST_DLLM_GENERATION_PROFILE)
    stale_rows = int((successful & ~current_profile).sum())
    if stale_rows:
        print(
            f"Ignoring {stale_rows} Fast-dLLM rows generated by the incompatible "
            "Transformers-5 adapter. They remain in the cache until replaced."
        )
    cached = cached.loc[successful & current_profile].copy()
    for column in ["output_tokens", "generation_s", "tokens_per_second"]:
        if column not in cached:
            cached[column] = np.nan
    audit = cached.merge(
        prompts[["prompt_id", "task", "prompt", "reference"]],
        on=["prompt_id", "task"], how="inner", validate="one_to_one",
    )
    if audit.empty:
        print("The Fast-dLLM cache contains no successful rows matching the current prompt set.")
        return audit

    audit["prediction"] = [extract_answer(r, t) for r, t in zip(audit.response, audit.task)]
    audit["parsed"] = audit.prediction.notna()
    audit["quality"] = audit.prediction.eq(audit.reference)
    audit["empty_response"] = audit.response.fillna("").astype(str).str.strip().eq("")
    audit["response_chars"] = audit.response.fillna("").astype(str).str.len()
    audit["at_token_cap"] = audit.output_tokens.ge(MAX_NEW_TOKENS)
    audit["degenerate_response"] = audit.response.map(is_degenerate_response)

    summary = audit.groupby("task").agg(
        cached_prompts=("prompt_id", "size"),
        parse_rate=("parsed", "mean"),
        exact_match=("quality", "mean"),
        empty_response_rate=("empty_response", "mean"),
        degenerate_response_rate=("degenerate_response", "mean"),
        token_cap_rate=("at_token_cap", "mean"),
        mean_output_tokens=("output_tokens", "mean"),
        mean_latency_s=("generation_s", "mean"),
        mean_tokens_per_second=("tokens_per_second", "mean"),
    )
    display(summary.round(3))

    checks = pd.Series({
        "all_three_tasks_present": set(audit.task) == {"gsm8k", "mmlu", "arc_challenge"},
        "parse_rate_at_least_90pct": audit.parsed.mean() >= 0.90,
        "at_least_one_exact_match": bool(audit.quality.any()),
        "no_degenerate_responses": not bool(audit.degenerate_response.any()),
        "token_cap_rate_at_most_25pct": audit.at_token_cap.mean() <= 0.25,
    }, name="passed")
    display(checks.rename_axis("audit_check").to_frame())
    print("Audit recommendation:", "PASS — expansion may proceed." if checks.all() else "HOLD — inspect responses before expansion.")

    sample_count = min(sample_per_task, int(audit.groupby("task").size().min()))
    sampled = audit.groupby("task", group_keys=False).sample(n=sample_count, random_state=SEED)
    with pd.option_context("display.max_colwidth", 1000, "display.max_rows", None):
        display(sampled[[
            "task", "prompt_id", "reference", "prediction", "parsed", "quality",
            "degenerate_response", "output_tokens", "generation_s", "response",
        ]].sort_values(["task", "prompt_id"]))

    audit_path = ROOT / "reports" / "fast_dllm_audit.parquet"
    audit.to_parquet(audit_path, index=False)
    print(f"Saved {len(audit)} audited Fast-dLLM rows to {audit_path}")
    if audit.parsed.mean() < 0.90:
        warnings.warn("Fast-dLLM parse coverage is below 90%; inspect raw responses before expanding.")
    if audit.quality.mean() == 0:
        warnings.warn("Fast-dLLM has zero exact matches; do not expand until decoding or parsing is fixed.")
    if audit.at_token_cap.mean() > 0.25:
        warnings.warn("More than 25% of Fast-dLLM responses hit MAX_NEW_TOKENS; inspect truncation.")
    if audit.degenerate_response.any():
        warnings.warn("Fast-dLLM still has repetitive low-diversity responses; do not expand.")
    return audit

### 4.2 Repair pilot, then generate missing prompt/model rows

The repair pilot regenerates five prompts per task using the official runtime;
legacy Fast-dLLM rows are ignored via a generation-profile tag. Inspect the new
audit before opening the expansion gate. Each full model run is loaded once,
warmed up, measured prompt by prompt, and saved resumably.

In [7]:

def save_cache(frame: pd.DataFrame, path: Path) -> None:
    clean = frame.drop_duplicates(["prompt_id", "model"], keep="last")
    temporary = path.with_suffix(".tmp.parquet")
    clean.to_parquet(temporary, index=False)
    temporary.replace(path)

def benchmark_model(model_name: str, prompt_frame: pd.DataFrame | None = None) -> pd.DataFrame:
    target_prompts = prompts if prompt_frame is None else prompt_frame
    path = cache_path(model_name)
    cached = pd.read_parquet(path) if path.exists() else pd.DataFrame()
    if cached.empty:
        completed = set()
    else:
        required_cache_columns = {"prompt_id", "model", "status"}
        missing_columns = required_cache_columns - set(cached.columns)
        if missing_columns:
            raise ValueError(f"Cache {path} is missing columns: {sorted(missing_columns)}")
        successful = cached["status"].eq("ok")
        if model_name == "fast-dllm-v2-1.5b":
            if "generation_profile" not in cached:
                cached["generation_profile"] = None
            successful &= cached.generation_profile.eq(FAST_DLLM_GENERATION_PROFILE)
        completed = set(cached.loc[successful, "prompt_id"])
    remaining = target_prompts.loc[~target_prompts.prompt_id.isin(completed)]
    print(f"{model_name}: {len(completed)} cached, {len(remaining)} remaining")
    if remaining.empty:
        return cached

    adapter = None
    try:
        adapter = LocalAdapter(model_name, MODELS[model_name])
        smoke = adapter.generate(remaining.iloc[0].prompt)
        assert smoke["response"] and smoke["output_tokens"] > 0
        if model_name == "fast-dllm-v2-1.5b" and is_degenerate_response(smoke["response"]):
            raise RuntimeError(
                "Fast-dLLM smoke test is still repetitive. Confirm Transformers 4.53.1 "
                "is active after restarting the Colab runtime."
            )
        print("smoke:", smoke["response"][:240].replace("\n", " "))
        for prompt in target_prompts.prompt.head(WARMUP_PROMPTS):
            adapter.generate(prompt)

        new_rows = []
        for run_index, row in enumerate(remaining.itertuples(), 1):
            base = {
                "prompt_id": row.prompt_id, "task": row.task, "model": model_name,
                "model_repo": MODELS[model_name].repo, "gpu": GPU_NAME,
                "load_time_s": adapter.load_time_s,
                "generation_profile": (
                    FAST_DLLM_GENERATION_PROFILE
                    if model_name == "fast-dllm-v2-1.5b"
                    else "standard-greedy-transformers-4.53.1"
                ),
            }
            try:
                result = {**base, **adapter.generate(row.prompt), "status": "ok", "error": None}
            except Exception as exc:
                result = {**base, "status": "error", "error": repr(exc)}
                new_rows.append(result)
                save_cache(pd.concat([cached, pd.DataFrame(new_rows)], ignore_index=True), path)
                raise
            new_rows.append(result)
            if run_index % FLUSH_EVERY == 0 or run_index == len(remaining):
                save_cache(pd.concat([cached, pd.DataFrame(new_rows)], ignore_index=True), path)
                print(f"  saved {run_index}/{len(remaining)}")
        return pd.read_parquet(path)
    finally:
        if adapter is not None:
            unload(adapter)
        else:
            gc.collect(); torch.cuda.empty_cache(); torch.cuda.synchronize()

repair_prompts = (
    prompts.sort_values("prompt_id")
    .groupby("task", group_keys=False)
    .head(FAST_REPAIR_PROMPTS_PER_TASK)
)
if RUN_FAST_DLLM_REPAIR_PILOT:
    benchmark_model("fast-dllm-v2-1.5b", repair_prompts)

fast_dllm_audit = audit_fast_dllm_cache()

if not RUN_EXPANDED_BENCHMARK:
    raise RuntimeError(
        "Audit gate is closed. Inspect fast_dllm_audit above, then set "
        "RUN_EXPANDED_BENCHMARK = True and rerun this cell to generate missing rows."
    )

for model_name in MODELS_TO_RUN:
    benchmark_model(model_name)

fast-dllm-v2-1.5b: 900 cached, 0 remaining


,cached_prompts,parse_rate,exact_match,empty_response_rate,degenerate_response_rate,token_cap_rate,mean_output_tokens,mean_latency_s,mean_tokens_per_second
task,,,,,,,,,
arc_challenge,300,0.997,0.460,0.0,0.0,0.007,134.387,4.007,34.095
gsm8k,300,1.000,0.317,0.0,0.0,0.030,160.773,2.994,55.994
mmlu,300,0.970,0.353,0.0,0.0,0.020,132.043,3.846,35.165


,passed
audit_check,
all_three_tasks_present,True
parse_rate_at_least_90pct,True
at_least_one_exact_match,True
no_degenerate_responses,True
token_cap_rate_at_most_25pct,True


Audit recommendation: PASS — expansion may proceed.


,task,prompt_id,reference,prediction,parsed,quality,degenerate_response,output_tokens,generation_s,response
752,arc_challenge,08b1b2bfff16fb79,B,D,True,False,False,137,4.923114,"The answer is D. The tapeworm helps the dog survive.\n\nTapeworms are parasites that live in the intestines of their hosts (dogs). They absorb nutrients from the food the dog eats, which can harm the dog's health. However, the tapeworms also benefit from this relationship. Tapeworms can help their hosts survive by absorbing nutrients from the food they consume, which can help the tapeworms' own growth and development. Therefore, the tapeworms help the dog survive, and the dog does not become ill as a result of this relationship.\n\nThe final answer is D. The tapeworm helps the dog survive."
803,arc_challenge,7819ca92f32a398a,C,C,True,True,False,44,1.223040,"The answer is C. food and nesting areas\n\nIn terrestrial biomes, most vertebrates compete for food and nesting areas. Carnores and herbores compete for food, while birds and mammals compete for nesting areas."
833,arc_challenge,cee78bb7f9cdedce,C,B,True,False,False,64,2.256000,"The state of a substance can be determined by knowing its density. Density is a measure of the amount of mass in a unit volume, and it is independent of the container. Therefore, to determine the state of the sample, we need to know its density.\n\nThe final answer is B. the density of the sample"
866,arc_challenge,e2e4fff7fe8f6c08,C,C,True,True,False,163,4.610103,"The answer is C. About 97 percent of Earth's water is ocean water.\n\nThe fact that explains why most evaporation happens in the oceans is that there is a significant amount of water in the oceans. This is because water is a good solvent, and it evaporates more easily than other substances. Additionally, the oceans cover about 71% of the Earth's surface, which provides a large area for water to evaporate. The other options do not directly explain the process of evaporation in the oceans. While ocean water is salty, this is not the primary reason for evaporation. There are mountain ranges on the ocean floor, but they do not affect evaporation. Fewer clouds blocking sunlight over the ocean is also not the main reason for evaporation. The final answer is C."
209,arc_challenge,fa208aab7dc362f5,A,A,True,True,False,70,2.148523,"The answer is A. Water can change phase.\n\nThe water cycle on Earth is possible due to the property of water to change its phase, specifically from liquid to gas and back. This property allows water to evaporate from the Earth's surface, rise into the atmosphere, and then condense and fall back to the Earth's surface as precipitation."
406,gsm8k,07a22f595904d9fd,28,28,True,True,False,127,2.552114,"Jeremy initially saw 12 birds in the backyard. He threw a stone at them, scaring away 1/3 of that number. So, the number of birds scared away is:\n\n12 * (1/3) = 4\n\nAfter throwing the stone, the number of birds remaining in the backyard is:\n\n12 - 4 = 8\n\nA few minutes later, 20 more birds joined the fearless birds. So, the total number of birds now in the backyard is:\n\n8 + 20 = 28\n\nTherefore, the answer is: {""answer"": 28}"
340,gsm8k,34c3ae985eb88442,2,36,True,False,False,178,2.165473,"To determine Millie's profit per hour, we need to follow these steps:\n\n1. Calculate the total number of cups of lemonade Mill makes.\n2. Determine the total revenue from selling the lemonade.\n3. Calculate the total cost of making the lemonade.\n4. Find the profit by subtracting the total cost from the total revenue.\n\n**Step 1: Calculate the total number of cups of lemonade Mill makes.**\n\nMillie makes 3 pitchers of lemonade, and each pitcher holds 12 cups.\n\[\n3 \text{ pitchers} \times 12 \text{ cups per pitcher} = 36 \text{ cups}\n\]\n\n**Step 2: Determine the total revenue from selling the lemonade.**\n\nMillie sells each cup of lemonade for $1.\n\[\n36 \text{ cups} \times"
456,gsm8k,80571773e6839755,68,12,True,False,False,187,4.563620,"Let's denote th

Saved 900 audited Fast-dLLM rows to /content/drive/MyDrive/llm_router_real/reports/fast_dllm_audit.parquet
qwen2.5-1.5b-ar: 900 cached, 0 remaining
fast-dllm-v2-1.5b: 900 cached, 0 remaining
qwen2.5-7b-4bit: 900 cached, 0 remaining


## 5. Deterministic quality labels

Accuracy is exact match after task-specific answer extraction. Inspect failures later: formatting mistakes are real failures for this benchmark, but the displayed error table lets you distinguish reasoning errors from parser errors.

In [8]:
missing = [name for name in MODELS if not cache_path(name).exists()]
if missing:
    raise RuntimeError(f"Run the missing candidates before analysis: {missing}")

parts = [pd.read_parquet(cache_path(name)) for name in MODELS]
measurements = pd.concat(parts, ignore_index=True)
measurements = measurements.drop_duplicates(["prompt_id", "model"], keep="last")
if "generation_profile" not in measurements:
    measurements["generation_profile"] = None
valid_fast_profile = (
    ~measurements.model.eq("fast-dllm-v2-1.5b")
    | measurements.generation_profile.eq(FAST_DLLM_GENERATION_PROFILE)
)
measurements = measurements.loc[valid_fast_profile]
measurements = measurements.loc[measurements.status.eq("ok")].merge(
    prompts, on=["prompt_id", "task"], validate="many_to_one"
)
expected_panel = len(prompts) * len(MODELS)
if len(measurements) != expected_panel:
    completed = measurements.groupby("model").prompt_id.nunique().reindex(MODELS, fill_value=0)
    raise RuntimeError(
        f"The prompt/model panel is incomplete: found {len(measurements)}/{expected_panel} rows.\n"
        f"Successful prompts by model:\n{completed.to_string()}"
    )
measurements["prediction"] = [extract_answer(r, t) for r, t in zip(measurements.response, measurements.task)]
measurements["quality"] = (measurements.prediction == measurements.reference).astype(float)
measurements["prompt_words"] = measurements.prompt.str.split().str.len()
measurements.to_parquet(ROOT / "data" / "measurements.parquet", index=False)

summary = measurements.groupby(["model", "task"]).agg(
    prompts=("prompt_id", "size"), accuracy=("quality", "mean"),
    latency_s=("generation_s", "mean"), output_tokens=("output_tokens", "mean"),
    tokens_per_second=("tokens_per_second", "mean"), peak_vram_gb=("peak_vram_gb", "max"),
)
display(summary.round(3))

prompts  accuracy  latency_s  output_tokens  \
model             task                                                         
fast-dllm-v2-1.5b arc_challenge      300     0.460      4.007        134.387   
                  gsm8k              300     0.317      2.994        160.773   
                  mmlu               300     0.353      3.846        132.043   
qwen2.5-1.5b-ar   arc_challenge      300     0.483      5.178        148.113   
                  gsm8k              300     0.347      6.569        184.037   
                  mmlu               300     0.430      4.785        137.920   
qwen2.5-7b-4bit   arc_challenge      300     0.890      0.862         12.737   
                  gsm8k              300     0.607      8.730        160.173   
                  mmlu               300     0.680      1.408         21.773   

                                 tokens_per_second  peak_vram_gb  
model             task                                            
fast-dllm-v2-1.5b arc_challenge             34.095         2.963  
                  gsm8k                     55.994         2.943  
                  mmlu                      35.165         3.204  
qwen2.5-1.5b-ar   arc_challenge             28.692         2.906  
                  gsm8k                     28.463         2.903  
                  mmlu                      28.885         2.978  
qwen2.5-7b-4bit   arc_challenge             12.939         5.355  
                  gsm8k                     18.363         5.346  
                  mmlu                      12.331         5.472

## 6. Fine-tuned ModernBERT router

This implements the architecture in `llm_router_project_scope_2.md`. A trainable
ModernBERT encoder consumes prompt text plus task metadata and feeds three heads:
candidate quality, log output-token count, and a soft-oracle policy distribution.
The policy target keeps only quality-eligible models and softly prefers lower
measured latency. Training and early stopping use only the train and validation
partitions; the test partition remains sealed until the final report.

In [9]:
MODEL_NAMES = list(MODELS)

def wide(column: str) -> pd.DataFrame:
    return measurements.pivot(index="prompt_id", columns="model", values=column).reindex(columns=MODEL_NAMES)

table = prompts.set_index("prompt_id").join(wide("quality").add_prefix("q__"))
table = table.join(wide("generation_s").add_prefix("l__"))
table = table.join(wide("output_tokens").add_prefix("t__")).reset_index()
table

,prompt_id,task,prompt,reference,q__qwen2.5-1.5b-ar,q__fast-dllm-v2-1.5b,q__qwen2.5-7b-4bit,l__qwen2.5-1.5b-ar,l__fast-dllm-v2-1.5b,l__qwen2.5-7b-4bit,t__qwen2.5-1.5b-ar,t__fast-dllm-v2-1.5b,t__qwen2.5-7b-4bit
0,58cf8075de61241b,gsm8k,"Solve the problem. Show concise reasoning, the...",36,0.0,1.0,1.0,6.795671,3.144231,7.480092,192,189,124
1,9de6593ad5d5b36d,gsm8k,"Solve the problem. Show concise reasoning, the...",189,0.0,0.0,0.0,14.427088,3.713410,10.890621,192,172,192
2,ce05d02f248f122a,gsm8k,"Solve the problem. Show concise reasoning, the...",65960,0.0,0.0,0.0,7.773379,2.752343,10.101326,192,162,192
3,6a75f556c1c6d990,gsm8k,"Solve the problem. Show concise reasoning, the...",30,1.0,0.0,1.0,7.225161,1.906294,7.378384,192,166,127
4,378447282c88ae35,gsm8k,"Solve the problem. Show concise reasoning, the...",25,0.0,0.0,1.0,13.255784,2.396746,11.030857,188,144,192
...,...,...,...,...,...,...,...,...,...,...,...,...,...
895,e5a16b4381117cfc,arc_challenge,"Choose the best answer. End with JSON {""answer...",A,1.0,1.0,1.0,5.698124,2.928004,0.507162,176,86,7
896,0d32bf5663b8016b,arc_challenge,"Choose the best answer. End with JSON {""answer...",B,0.0,1.0,0.0,3.812454,5.180410,0.609145,120,168,7
897,3ad71343f3327207,arc_challenge,"Choose the best answer. End with JSON {""answer...",B,0.0,0.0,0.0,7.123509,3.093716,10.534662,192,175,192
898,3df017dfcbbf0b5a,arc_challenge,"Choose the best answer. End with JSON {""answer...",B,0.0,0.0,1.0,6.022302,4.798983,0.526195,192,159,7


In [10]:
MODEL_NAMES = list(MODELS)

def wide(column: str) -> pd.DataFrame:
    return measurements.pivot(index="prompt_id", columns="model", values=column).reindex(columns=MODEL_NAMES)

table = prompts.set_index("prompt_id").join(wide("quality").add_prefix("q__"))
table = table.join(wide("generation_s").add_prefix("l__"))
table = table.join(wide("output_tokens").add_prefix("t__")).reset_index()

train_val, test = train_test_split(table.index, test_size=.20, random_state=SEED, stratify=table.task)
train, validation = train_test_split(
    train_val, test_size=.25, random_state=SEED + 1, stratify=table.loc[train_val, "task"]
)
masks = {"train": table.index.isin(train), "validation": table.index.isin(validation), "test": table.index.isin(test)}

Q = table[[f"q__{m}" for m in MODEL_NAMES]].to_numpy(float)
L = table[[f"l__{m}" for m in MODEL_NAMES]].to_numpy(float)
T = table[[f"t__{m}" for m in MODEL_NAMES]].to_numpy(float)
actual_best = Q.max(axis=1, keepdims=True)
actual_eligible = Q >= actual_best - ORACLE_EPSILON
oracle_idx = np.where(actual_eligible, L, np.inf).argmin(axis=1)

text = ("[TASK=" + table.task + "] " + table.prompt).to_numpy()
from transformers import AutoModel, get_cosine_schedule_with_warmup

ROUTER_ENCODER_REPO = "nomic-ai/modernbert-embed-base"
ROUTER_ENCODER_REVISION = "d556a88e332558790b210f7bdbe87da2fa94a8d8"
ROUTER_MAX_INPUT_TOKENS = MAX_INPUT_TOKENS
ROUTER_WARMUP_PROMPTS = min(2, len(text))

def build_soft_oracle_targets() -> np.ndarray:
    """Create quality-gated soft targets using train-derived latency scaling."""
    train_latency = L[masks["train"]]
    latency_mean = float(train_latency.mean())
    latency_scale = float(train_latency.std()) or 1.0
    normalized_latency = (L - latency_mean) / latency_scale
    score = Q - SOFT_ORACLE_LATENCY_WEIGHT * normalized_latency
    score = np.where(actual_eligible, score, -np.inf)
    row_max = np.max(score, axis=1, keepdims=True)
    weights = np.where(
        actual_eligible,
        np.exp((score - row_max) / SOFT_ORACLE_TEMPERATURE),
        0.0,
    )
    return weights / weights.sum(axis=1, keepdims=True)

soft_oracle = build_soft_oracle_targets()
quality_targets = torch.tensor(Q, dtype=torch.float32)
token_targets = torch.tensor(np.log1p(T), dtype=torch.float32)
policy_targets = torch.tensor(soft_oracle, dtype=torch.float32)

router_load_started = time.perf_counter()
router_tokenizer = AutoTokenizer.from_pretrained(
    ROUTER_ENCODER_REPO, revision=ROUTER_ENCODER_REVISION
)

class ModernBERTRouter(torch.nn.Module):
    """Trainable ModernBERT with quality, token-count, and policy heads."""

    def __init__(self):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(
            ROUTER_ENCODER_REPO, revision=ROUTER_ENCODER_REVISION,
            attn_implementation="sdpa",
        )
        hidden_size = int(self.encoder.config.hidden_size)
        self.dropout = torch.nn.Dropout(0.10)
        self.quality_head = torch.nn.Linear(hidden_size, len(MODEL_NAMES))
        self.token_head = torch.nn.Linear(hidden_size, len(MODEL_NAMES))
        self.policy_head = torch.nn.Linear(hidden_size, len(MODEL_NAMES))

    def forward(self, **inputs) -> dict[str, torch.Tensor]:
        hidden = self.encoder(**inputs).last_hidden_state
        mask = inputs["attention_mask"].unsqueeze(-1).to(hidden.dtype)
        pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp_min(1.0)
        pooled = self.dropout(pooled)
        return {
            "quality": torch.sigmoid(self.quality_head(pooled)),
            "token_log": self.token_head(pooled),
            "policy_logits": self.policy_head(pooled),
        }

router_model = ModernBERTRouter().to("cuda")
router_load_time_s = time.perf_counter() - router_load_started

def collate_router_batch(indices: list[int]):
    batch_indices = torch.tensor(indices, dtype=torch.long)
    encoded = router_tokenizer(
        [f"classification: {text[i]}" for i in indices],
        padding=True, truncation=True, max_length=ROUTER_MAX_INPUT_TOKENS,
        return_tensors="pt",
    )
    return batch_indices, encoded

def make_router_loader(mask: np.ndarray, shuffle: bool) -> DataLoader:
    indices = np.flatnonzero(mask).tolist()
    generator = torch.Generator().manual_seed(SEED)
    return DataLoader(
        indices, batch_size=ROUTER_BATCH_SIZE, shuffle=shuffle,
        collate_fn=collate_router_batch, generator=generator,
    )

train_loader = make_router_loader(masks["train"], shuffle=True)
validation_loader = make_router_loader(masks["validation"], shuffle=False)

def router_batch_loss(outputs: dict[str, torch.Tensor], indices: torch.Tensor):
    q_target = quality_targets[indices].to("cuda")
    t_target = token_targets[indices].to("cuda")
    p_target = policy_targets[indices].to("cuda")
    quality_loss = F.mse_loss(outputs["quality"].float(), q_target)
    token_loss = F.smooth_l1_loss(outputs["token_log"].float(), t_target)
    policy_loss = -(
        p_target * F.log_softmax(outputs["policy_logits"].float(), dim=1)
    ).sum(dim=1).mean()
    total = (
        ROUTER_QUALITY_LOSS_WEIGHT * quality_loss
        + ROUTER_TOKEN_LOSS_WEIGHT * token_loss
        + ROUTER_POLICY_LOSS_WEIGHT * policy_loss
    )
    return total, quality_loss, token_loss, policy_loss

encoder_parameters = list(router_model.encoder.parameters())
head_parameters = [
    parameter for name, parameter in router_model.named_parameters()
    if not name.startswith("encoder.")
]
optimizer = torch.optim.AdamW([
    {"params": encoder_parameters, "lr": ROUTER_ENCODER_LR},
    {"params": head_parameters, "lr": ROUTER_HEAD_LR},
], weight_decay=ROUTER_WEIGHT_DECAY)
updates_per_epoch = math.ceil(len(train_loader) / ROUTER_GRADIENT_ACCUMULATION)
total_training_steps = updates_per_epoch * ROUTER_MAX_EPOCHS
warmup_steps = max(1, round(total_training_steps * ROUTER_WARMUP_RATIO))
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_training_steps,
)
scaler = torch.amp.GradScaler("cuda", enabled=COMPUTE_DTYPE == torch.float16)

@torch.inference_mode()
def evaluate_router_loss(loader: DataLoader) -> dict[str, float]:
    router_model.eval()
    totals = np.zeros(4, dtype=float); examples = 0
    for indices, encoded in loader:
        encoded = encoded.to("cuda")
        with torch.autocast("cuda", dtype=COMPUTE_DTYPE):
            losses = router_batch_loss(router_model(**encoded), indices)
        batch_size = len(indices); examples += batch_size
        totals += batch_size * np.array([float(loss.detach()) for loss in losses])
    return dict(zip(["loss", "quality_loss", "token_loss", "policy_loss"], totals / examples))

history = []
best_validation_loss = np.inf
best_state = None
best_epoch = 0
epochs_without_improvement = 0
training_started = time.perf_counter()

for epoch in range(1, ROUTER_MAX_EPOCHS + 1):
    router_model.train(); optimizer.zero_grad(set_to_none=True)
    train_totals = np.zeros(4, dtype=float); train_examples = 0
    for step, (indices, encoded) in enumerate(train_loader, 1):
        encoded = encoded.to("cuda")
        with torch.autocast("cuda", dtype=COMPUTE_DTYPE):
            losses = router_batch_loss(router_model(**encoded), indices)
            scaled_loss = losses[0] / ROUTER_GRADIENT_ACCUMULATION
        scaler.scale(scaled_loss).backward()
        if step % ROUTER_GRADIENT_ACCUMULATION == 0 or step == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                router_model.parameters(), ROUTER_MAX_GRAD_NORM
            )
            scaler.step(optimizer); scaler.update(); scheduler.step()
            optimizer.zero_grad(set_to_none=True)
        batch_size = len(indices); train_examples += batch_size
        train_totals += batch_size * np.array([float(loss.detach()) for loss in losses])

    validation_losses = evaluate_router_loss(validation_loader)
    row = {
        "epoch": epoch,
        "encoder_lr": optimizer.param_groups[0]["lr"],
        "head_lr": optimizer.param_groups[1]["lr"],
    }
    row.update({f"train_{name}": value for name, value in zip(
        ["loss", "quality_loss", "token_loss", "policy_loss"],
        train_totals / train_examples,
    )})
    row.update({f"validation_{name}": value for name, value in validation_losses.items()})
    history.append(row)
    print({key: round(value, 4) if isinstance(value, float) else value for key, value in row.items()})

    if validation_losses["loss"] < (
        best_validation_loss - ROUTER_EARLY_STOPPING_MIN_DELTA
    ):
        best_validation_loss = validation_losses["loss"]
        best_epoch = epoch
        best_state = {name: value.detach().cpu().clone() for name, value in router_model.state_dict().items()}
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
        if (
            epoch >= ROUTER_MIN_EPOCHS
            and epochs_without_improvement >= ROUTER_EARLY_STOPPING_PATIENCE
        ):
            print(f"Early stopping after epoch {epoch}; restoring epoch {best_epoch}.")
            break

router_training_time_s = time.perf_counter() - training_started
assert best_state is not None
router_model.load_state_dict(best_state)
del best_state
training_history = pd.DataFrame(history)
training_history.to_csv(ROOT / "reports" / "router_training_history.csv", index=False)
display(training_history.round(4))

@torch.inference_mode()
def predict_with_router(texts: np.ndarray):
    """Predict all three heads and measure online batch-size-one router latency."""
    router_model.eval()

    def predict_one(value: str):
        encoded = router_tokenizer(
            f"classification: {value}", return_tensors="pt", truncation=True,
            max_length=ROUTER_MAX_INPUT_TOKENS,
        ).to("cuda")
        with torch.autocast("cuda", dtype=COMPUTE_DTYPE):
            return router_model(**encoded)

    for value in texts[:ROUTER_WARMUP_PROMPTS]:
        predict_one(str(value))
    torch.cuda.synchronize()

    qualities, token_logs, policies = [], [], []
    latency_s = np.zeros(len(texts), dtype=float)
    for i, value in enumerate(texts):
        torch.cuda.synchronize(); started = time.perf_counter()
        outputs = predict_one(str(value))
        torch.cuda.synchronize(); latency_s[i] = time.perf_counter() - started
        qualities.append(outputs["quality"].float().cpu().numpy())
        token_logs.append(outputs["token_log"].float().cpu().numpy())
        policies.append(F.softmax(outputs["policy_logits"].float(), dim=1).cpu().numpy())
    return (
        np.concatenate(qualities), np.concatenate(token_logs),
        np.concatenate(policies), latency_s,
    )

quality_pred, token_log_pred, policy_prob, router_feature_latency_s = predict_with_router(text)
token_pred = np.maximum(1.0, np.expm1(token_log_pred))
print({
    "router_encoder": ROUTER_ENCODER_REPO,
    "router_encoder_revision": ROUTER_ENCODER_REVISION,
    "trainable_parameters": sum(parameter.numel() for parameter in router_model.parameters()),
    "best_epoch": best_epoch,
    "load_time_s": round(router_load_time_s, 3),
    "training_time_s": round(router_training_time_s, 3),
    "mean_online_router_ms": round(1000 * router_feature_latency_s.mean(), 3),
    "train_prompts": int(masks["train"].sum()),
})

del router_model, router_tokenizer, optimizer, scheduler, scaler
gc.collect(); torch.cuda.empty_cache(); torch.cuda.synchronize()

latency_pred = np.zeros_like(L)
latency_heads = {}
latency_diagnostics = []
for j, name in enumerate(MODEL_NAMES):
    features = T[masks["train"], j].reshape(-1, 1)
    head = LinearRegression().fit(features, L[masks["train"], j])
    latency_heads[name] = head
    latency_pred[:, j] = np.maximum(1e-6, head.predict(token_pred[:, j].reshape(-1, 1)))
    latency_diagnostics.append({
        "model": name, "intercept_s": float(head.intercept_),
        "seconds_per_output_token": float(head.coef_[0]),
        "train_r2": float(head.score(features, L[masks["train"], j])),
    })
display(pd.DataFrame(latency_diagnostics).set_index("model").round(4))

strongest_idx = int(Q[masks["train"]].mean(axis=0).argmax())
fastest_idx = int(L[masks["train"]].mean(axis=0).argmin())
print("Strongest fallback:", MODEL_NAMES[strongest_idx], "| Fastest baseline:", MODEL_NAMES[fastest_idx])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

{'epoch': 1, 'encoder_lr': 0.0, 'head_lr': 0.0001, 'train_loss': np.float64(1.8209), 'train_quality_loss': np.float64(0.2396), 'train_token_loss': np.float64(2.8615), 'train_policy_loss': np.float64(1.009), 'validation_loss': np.float64(1.4514), 'validation_quality_loss': np.float64(0.2362), 'validation_token_loss': np.float64(0.8864), 'validation_policy_loss': np.float64(1.0379)}
{'epoch': 2, 'encoder_lr': 0.0, 'head_lr': 0.0001, 'train_loss': np.float64(1.3149), 'train_quality_loss': np.float64(0.2307), 'train_token_loss': np.float64(0.3772), 'train_policy_loss': np.float64(1.0088), 'validation_loss': np.float64(1.2408), 'validation_quality_loss': np.float64(0.2299), 'validation_token_loss': np.float64(0.1906), 'validation_policy_loss': np.float64(0.9728)}
{'epoch': 3, 'encoder_lr': 0.0, 'head_lr': 0.0001, 'train_loss': np.float64(1.2423), 'train_quality_loss': np.float64(0.221), 'train_token_loss': np.float64(0.1767), 'train_policy_loss': np.float64(0.9859), 'validation_loss': np.fl

,epoch,encoder_lr,head_lr,train_loss,train_quality_loss,train_token_loss,train_policy_loss,validation_loss,validation_quality_loss,validation_token_loss,validation_policy_loss
0,1,0.0,0.0001,1.8209,0.2396,2.8615,1.0090,1.4514,0.2362,0.8864,1.0379
1,2,0.0,0.0001,1.3149,0.2307,0.3772,1.0088,1.2408,0.2299,0.1906,0.9728
2,3,0.0,0.0001,1.2423,0.2210,0.1767,0.9859,1.2694,0.2267,0.1468,1.0134
3,4,0.0,0.0001,1.2157,0.2190,0.1627,0.9642,1.2564,0.2288,0.1614,0.9953
4,5,0.0,0.0001,1.1952,0.2141,0.1627,0.9485,1.2216,0.2239,0.1464,0.9684
5,6,0.0,0.0001,1.1662,0.2093,0.1643,0.9241,1.2445,0.2231,0.1578,0.9899
6,7,0.0,0.0001,1.1145,0.2016,0.1574,0.8815,1.2244,0.2195,0.1538,0.9741
7,8,0.0,0.0001,1.0750,0.1944,0.1632,0.8480,1.2543,0.2217,0.1849,0.9956
8,9,0.0,0.0001,1.0189,0.1851,0.1687,0.8000,1.2496,0.2207,0.1754,0.9939
9,10,0.0,0.0001,0.9530,0.1716,0.1559,0.7502,1.2464,0.2186,0.1784,0.9921


{'router_encoder': 'nomic-ai/modernbert-embed-base', 'router_encoder_revision': 'd556a88e332558790b210f7bdbe87da2fa94a8d8', 'trainable_parameters': 149021193, 'best_epoch': 5, 'load_time_s': 27.363, 'training_time_s': 187.043, 'mean_online_router_ms': np.float64(30.307), 'train_prompts': 540}


,intercept_s,seconds_per_output_token,train_r2
model,,,
qwen2.5-1.5b-ar,0.0498,0.0351,0.8702
fast-dllm-v2-1.5b,0.6250,0.0211,0.4556
qwen2.5-7b-4bit,0.2099,0.0533,0.9961


Strongest fallback: qwen2.5-7b-4bit | Fastest baseline: qwen2.5-7b-4bit


## 7. Validation calibration and sealed test

Validation selects the fastest configuration meeting 98% aggregate quality retention relative to the strongest model. Test additionally reports the per-prompt quality-constraint violation rate and 95% bootstrap intervals; aggregate retention alone can hide bad individual routes.

In [11]:
def select(epsilon: float, confidence_threshold: float):
    eligible = quality_pred >= quality_pred.max(axis=1, keepdims=True) - epsilon
    chosen = np.where(eligible, latency_pred, np.inf).argmin(axis=1)
    confidence = policy_prob[np.arange(len(chosen)), chosen]
    used_fallback = confidence < confidence_threshold
    chosen[used_fallback] = strongest_idx
    return chosen, confidence, used_fallback

def metrics(mask, chosen, overhead_s=None) -> dict:
    rows = np.flatnonzero(mask); pick = chosen[mask]
    chosen_q = Q[mask][np.arange(len(rows)), pick]
    generation_l = L[mask][np.arange(len(rows)), pick]
    route_overhead = np.zeros(len(rows)) if overhead_s is None else np.asarray(overhead_s)[mask]
    chosen_l = generation_l + route_overhead
    strong_q, strong_l = Q[mask, strongest_idx], L[mask, strongest_idx]
    regret = Q[mask].max(axis=1) - chosen_q
    return {
        "accuracy": chosen_q.mean(),
        "quality_retention": chosen_q.mean() / max(strong_q.mean(), 1e-9),
        "latency_s": chosen_l.mean(),
        "router_overhead_s": route_overhead.mean(),
        "latency_reduction": 1 - chosen_l.mean() / strong_l.mean(),
        "strongest_usage": np.mean(pick == strongest_idx),
        "constraint_violation_rate": np.mean(regret > ORACLE_EPSILON),
        "p95_quality_regret": np.quantile(regret, .95),
    }

grid = []
for epsilon in [0, .025, .05, .10, .15, .20]:
    for threshold in [0, .25, .40, .55, .70, .85, .95]:
        candidate_idx, _, _ = select(epsilon, threshold)
        grid.append({"epsilon": epsilon, "confidence_threshold": threshold,
                     **metrics(
                         masks["validation"], candidate_idx,
                         router_feature_latency_s,
                     )})
calibration = pd.DataFrame(grid)
display(calibration.sort_values("latency_reduction", ascending=False).head(10).round(4))
feasible = calibration.query(
    "quality_retention >= @MIN_QUALITY_RETENTION and latency_reduction > 0"
)
if feasible.empty:
    warnings.warn(
        "No validation setting improved latency while meeting retention; "
        "deploying the strongest model without router overhead."
    )
    best = pd.Series({"epsilon": 0.0, "confidence_threshold": 1.0})
    router_idx = np.full(len(table), strongest_idx)
    router_confidence = policy_prob[:, strongest_idx]
    router_used_fallback = np.ones(len(table), dtype=bool)
    router_active = False
    selected_router_overhead_s = None  # The fallback policy does not run ModernBERT.
else:
    best = feasible.sort_values(
        ["latency_reduction", "constraint_violation_rate"], ascending=[False, True]
    ).iloc[0]
    router_idx, router_confidence, router_used_fallback = select(
        float(best.epsilon), float(best.confidence_threshold)
    )
    router_active = True
    selected_router_overhead_s = router_feature_latency_s

strategies = {
    "strongest": (np.full(len(table), strongest_idx), None),
    "fastest": (np.full(len(table), fastest_idx), None),
    "oracle": (oracle_idx, None),
    "router": (router_idx, selected_router_overhead_s),
}
evaluation = pd.DataFrame({
    name: metrics(masks["test"], pick, overhead)
    for name, (pick, overhead) in strategies.items()
}).T
display(evaluation.round(4))
print("Calibrated selector:", {
    "active": router_active,
    "epsilon": float(best.epsilon),
    "confidence_threshold": float(best.confidence_threshold),
})

,epsilon,confidence_threshold,accuracy,quality_retention,latency_s,router_overhead_s,latency_reduction,strongest_usage,constraint_violation_rate,p95_quality_regret
35,0.20,0.00,0.6722,0.9918,3.9025,0.0305,0.0123,0.9778,0.1000,1.0
36,0.20,0.25,0.6778,1.0000,3.9383,0.0305,0.0033,0.9889,0.0944,1.0
38,0.20,0.55,0.6778,1.0000,3.9418,0.0305,0.0024,0.9944,0.0944,1.0
37,0.20,0.40,0.6778,1.0000,3.9418,0.0305,0.0024,0.9944,0.0944,1.0
28,0.15,0.00,0.6722,0.9918,3.9701,0.0305,-0.0048,0.9944,0.1000,1.0
4,0.00,0.70,0.6778,1.0000,3.9817,0.0305,-0.0077,1.0000,0.0944,1.0
2,0.00,0.40,0.6778,1.0000,3.9817,0.0305,-0.0077,1.0000,0.0944,1.0
3,0.00,0.55,0.6778,1.0000,3.9817,0.0305,-0.0077,1.0000,0.0944,1.0
0,0.00,0.00,0.6778,1.0000,3.9817,0.0305,-0.0077,1.0000,0.0944,1.0
1,0.00,0.25,0.6778,1.0000,3.9817,0.0305,-0.0077,1.0000,0.0944,1.0


,accuracy,quality_retention,latency_s,router_overhead_s,latency_reduction,strongest_usage,constraint_violation_rate,p95_quality_regret
strongest,0.7333,1.0000,3.7414,0.0000,0.0000,1.0000,0.0500,0.05
fastest,0.7333,1.0000,3.7414,0.0000,0.0000,1.0000,0.0500,0.05
oracle,0.7833,1.0682,2.1792,0.0000,0.4175,0.6500,0.0000,0.00
router,0.7278,0.9924,3.7280,0.0299,0.0036,0.9889,0.0556,1.00


Calibrated selector: {'active': True, 'epsilon': 0.2, 'confidence_threshold': 0.0}


In [12]:
test_rows = np.flatnonzero(masks["test"])
test_pick = router_idx[masks["test"]]
report = table.loc[masks["test"], ["prompt_id", "task", "prompt"]].copy()
report["selected_model"] = [MODEL_NAMES[i] for i in test_pick]
report["quality"] = Q[masks["test"]][np.arange(len(test_rows)), test_pick]
report["generation_s"] = L[masks["test"]][np.arange(len(test_rows)), test_pick]
report["router_overhead_s"] = (
    0.0 if selected_router_overhead_s is None
    else selected_router_overhead_s[masks["test"]]
)
report["latency_s"] = report.generation_s + report.router_overhead_s
report["quality_regret"] = Q[masks["test"]].max(axis=1) - report.quality.to_numpy()
report["confidence"] = router_confidence[masks["test"]]
report["used_fallback"] = router_used_fallback[masks["test"]]
report.to_parquet(ROOT / "reports" / "test_decisions.parquet", index=False)
evaluation.to_csv(ROOT / "reports" / "evaluation.csv")

rng = np.random.default_rng(SEED)
boot = []
router_q = report.quality.to_numpy(); router_l = report.latency_s.to_numpy()
strong_q = Q[masks["test"], strongest_idx]; strong_l = L[masks["test"], strongest_idx]
for _ in range(2000):
    ix = rng.integers(0, len(report), len(report))
    boot.append([router_q[ix].mean() - strong_q[ix].mean(), 1 - router_l[ix].mean() / strong_l[ix].mean()])
ci = pd.DataFrame(boot, columns=["accuracy_delta", "latency_reduction"]).quantile([.025, .5, .975])
display(ci.round(4))
display(pd.crosstab(report.task, report.selected_model, normalize="index").round(3))
display(report.sort_values("quality_regret", ascending=False).head(10))

,accuracy_delta,latency_reduction
0.025,-0.0167,-0.0088
0.500,-0.0056,0.0034
0.975,0.0000,0.0212


selected_model,qwen2.5-1.5b-ar,qwen2.5-7b-4bit
task,,
arc_challenge,0.000,1.000
gsm8k,0.033,0.967
mmlu,0.000,1.000


,prompt_id,task,prompt,selected_model,quality,generation_s,router_overhead_s,latency_s,quality_regret,confidence,used_fallback
225,1645fe940088c07d,gsm8k,"Solve the problem. Show concise reasoning, the...",qwen2.5-7b-4bit,0.0,10.355237,0.035643,10.390881,1.0,0.527105,False
157,45140ad929ff03d5,gsm8k,"Solve the problem. Show concise reasoning, the...",qwen2.5-7b-4bit,0.0,10.435417,0.028594,10.464011,1.0,0.364003,False
134,c51ddf5d2f893f40,gsm8k,"Solve the problem. Show concise reasoning, the...",qwen2.5-7b-4bit,0.0,10.554761,0.028420,10.583180,1.0,0.335120,False
98,8636a9bd1ed2ff31,gsm8k,"Solve the problem. Show concise reasoning, the...",qwen2.5-1.5b-ar,0.0,6.742438,0.027087,6.769525,1.0,0.107401,False
443,bbaddd6cd6698729,mmlu,"Choose the best answer. End with JSON {""answer...",qwen2.5-7b-4bit,0.0,0.747020,0.031459,0.778479,1.0,0.557442,False
448,3b588ad1e89a2f22,mmlu,"Choose the best answer. End with JSON {""answer...",qwen2.5-7b-4bit,0.0,0.518070,0.026510,0.544580,1.0,0.649492,False
658,e7632727c61dec50,arc_challenge,"Choose the best answer. End with JSON {""answer...",qwen2.5-7b-4bit,0.0,0.537317,0.027699,0.565017,1.0,0.648333,False
463,2887e11b75d27660,mmlu,"Choose the best answer. End with JSON {""answer...",qwen2.5-7b-4bit,0.0,0.603543,0.029218,0.632761,1.0,0.676247,False
433,85b86764ea4589f6,mmlu,"Choose the best answer. End with JSON {""answer...",qwen2.5-7b-4bit,0.0,0.494506,0.029010,0.523516,1.0,0.580029,False
440,e91494919d0ce1a2,mmlu,"Choose the best answer. End with JSON {""answer...",qwen2.5-7b-4bit,0.0,0.597930,0.027568,0.625498,1.0,0.590213,False


## 8. Interactive prompt-by-prompt inspection

Use the task selector and prompt slider to review all 100 prompts in each task.
Each card shows the router decision, measured and predicted metrics, and the raw
response from every candidate. Train and validation decisions are diagnostic;
only the test rows contribute to the sealed result above.

In [13]:
import ipywidgets as widgets

split_name = np.full(len(table), "", dtype=object)
for name, mask in masks.items():
    split_name[mask] = name

all_rows = np.arange(len(table))
inspection_decisions = table[["prompt_id", "task", "prompt", "reference"]].copy()
inspection_decisions["split"] = split_name
inspection_decisions["selected_model"] = [MODEL_NAMES[index] for index in router_idx]
inspection_decisions["actual_quality"] = Q[all_rows, router_idx]
inspection_decisions["measured_generation_s"] = L[all_rows, router_idx]
inspection_decisions["predicted_quality"] = quality_pred[all_rows, router_idx]
inspection_decisions["predicted_generation_s"] = latency_pred[all_rows, router_idx]
inspection_decisions["policy_confidence"] = router_confidence
inspection_decisions["used_fallback"] = router_used_fallback
inspection_decisions.to_parquet(ROOT / "reports" / "all_prompt_decisions.parquet", index=False)

responses_by_prompt = {
    prompt_id: frame.set_index("model")
    for prompt_id, frame in measurements.groupby("prompt_id", sort=False)
}
task_rows = {
    task: frame.sort_values("prompt_id").reset_index(drop=True)
    for task, frame in inspection_decisions.groupby("task", sort=True)
}

task_selector = widgets.Dropdown(
    options=list(task_rows), description="Task:", layout=widgets.Layout(width="330px")
)
prompt_selector = widgets.IntSlider(
    value=1, min=1, max=N_PER_TASK, step=1, description="Prompt:",
    continuous_update=False, layout=widgets.Layout(width="520px"),
)
previous_button = widgets.Button(description="← Previous")
next_button = widgets.Button(description="Next →", button_style="primary")
inspector_output = widgets.Output()

def format_metric(value, digits=3):
    return "—" if pd.isna(value) else f"{float(value):.{digits}f}"

def render_prompt_card(*_):
    frame = task_rows[task_selector.value]
    position = min(prompt_selector.value - 1, len(frame) - 1)
    row = frame.iloc[position]
    candidates = responses_by_prompt[row.prompt_id]
    selected = row.selected_model
    table_index = int(table.index[table.prompt_id.eq(row.prompt_id)][0])

    comparison_rows = []
    response_sections = []
    for model_index, model_name in enumerate(MODEL_NAMES):
        measured = candidates.loc[model_name]
        chosen_class = "chosen" if model_name == selected else ""
        marker = "✓ Selected" if model_name == selected else ""
        comparison_rows.append(
            f'<tr class="{chosen_class}"><td><strong>{html_lib.escape(model_name)}</strong><br>'
            f'<span class="selected-marker">{marker}</span></td>'
            f'<td>{int(measured.quality)}</td>'
            f'<td>{format_metric(quality_pred[table_index, model_index])}</td>'
            f'<td>{format_metric(measured.generation_s)} s</td>'
            f'<td>{format_metric(latency_pred[table_index, model_index])} s</td>'
            f'<td>{int(measured.output_tokens)}</td></tr>'
        )
        response_sections.append(
            f'<details {"open" if model_name == selected else ""}>'
            f'<summary>{"★ " if model_name == selected else ""}{html_lib.escape(model_name)} response</summary>'
            f'<pre>{html_lib.escape(str(measured.response))}</pre></details>'
        )

    fallback_label = "Yes" if bool(row.used_fallback) else "No"
    card = f"""
    <style>
      .router-card {{font-family:Inter,system-ui,sans-serif;border:1px solid #dbe3ef;border-radius:18px;
        padding:22px;background:linear-gradient(145deg,#ffffff,#f6f9ff);box-shadow:0 8px 28px #25385818;color:#172033}}
      .router-card .top {{display:flex;gap:10px;align-items:center;flex-wrap:wrap;margin-bottom:14px}}
      .router-card .pill {{padding:5px 10px;border-radius:999px;background:#e8efff;color:#294fb5;font-size:12px;font-weight:700}}
      .router-card .model {{background:#dff7ec;color:#116149}}
      .router-card h3 {{margin:8px 0 6px;font-size:20px}}
      .router-card .prompt {{white-space:pre-wrap;background:#fff;border-left:4px solid #668cff;padding:14px;border-radius:8px;margin:12px 0}}
      .router-card .metrics {{display:grid;grid-template-columns:repeat(auto-fit,minmax(150px,1fr));gap:10px;margin:14px 0}}
      .router-card .metric {{background:#fff;border:1px solid #e5eaf2;border-radius:10px;padding:10px}}
      .router-card .metric small {{display:block;color:#68758a}}
      .router-card table {{border-collapse:collapse;width:100%;background:#fff;margin:14px 0}}
      .router-card th,.router-card td {{padding:9px;border-bottom:1px solid #e8edf5;text-align:left}}
      .router-card th {{color:#59677c;font-size:12px;text-transform:uppercase}}
      .router-card tr.chosen {{background:#edf9f4}}
      .router-card .selected-marker {{color:#14805e;font-size:11px;font-weight:700}}
      .router-card details {{background:#fff;border:1px solid #e5eaf2;border-radius:10px;padding:10px 12px;margin-top:8px}}
      .router-card summary {{cursor:pointer;font-weight:700}}
      .router-card pre {{white-space:pre-wrap;max-height:360px;overflow:auto;color:#273349;font-family:ui-monospace,monospace}}
    </style>
    <div class="router-card">
      <div class="top"><span class="pill">{html_lib.escape(str(row.task))}</span>
        <span class="pill">{html_lib.escape(str(row.split)).upper()}</span>
        <span class="pill model">Selected: {html_lib.escape(selected)}</span></div>
      <h3>Prompt {position + 1} of {len(frame)}</h3>
      <div class="prompt">{html_lib.escape(str(row.prompt))}</div>
      <div class="metrics">
        <div class="metric"><small>Reference</small><strong>{html_lib.escape(str(row.reference))}</strong></div>
        <div class="metric"><small>Actual quality</small><strong>{int(row.actual_quality)}</strong></div>
        <div class="metric"><small>Policy confidence</small><strong>{format_metric(row.policy_confidence)}</strong></div>
        <div class="metric"><small>Fallback used</small><strong>{fallback_label}</strong></div>
      </div>
      <table><thead><tr><th>Candidate</th><th>Actual quality</th><th>Predicted quality</th>
        <th>Measured latency</th><th>Predicted latency</th><th>Output tokens</th></tr></thead>
        <tbody>{''.join(comparison_rows)}</tbody></table>
      {''.join(response_sections)}
    </div>
    """
    with inspector_output:
        clear_output(wait=True)
        display(HTML(card))

def reset_task(change):
    prompt_selector.max = len(task_rows[change["new"]])
    prompt_selector.value = 1
    render_prompt_card()

def previous_prompt(_):
    prompt_selector.value = max(prompt_selector.min, prompt_selector.value - 1)

def next_prompt(_):
    prompt_selector.value = min(prompt_selector.max, prompt_selector.value + 1)

task_selector.observe(reset_task, names="value")
prompt_selector.observe(render_prompt_card, names="value")
previous_button.on_click(previous_prompt)
next_button.on_click(next_prompt)
display(widgets.VBox([
    widgets.HBox([task_selector, previous_button, next_button]),
    prompt_selector,
    inspector_output,
]))
render_prompt_card()

## 9. Reading the result

A positive result requires all of the following:

1. the router's test quality retention is at least 0.98;
2. latency reduction is positive and its bootstrap interval is reasonably stable;
3. constraint violations and high-regret examples are acceptable for the application;
4. more than one model is selected for defensible prompt subgroups.

This pilot estimates **controlled warm inference**, not single-GPU production routing: an online router only saves time when candidate workers are already loaded. Increase sample size, repeat seeds, and add a sandboxed code benchmark before making production claims.